# Setup: Postgres Batch-Read Benchmark Table (`bench_patch_batch_read`)

## Schema Description

This notebook creates and seeds the `bench_patch_batch_read` unlogged table used for the
**Batch read of patch data for model training (Postgres)** benchmark.

The schema mirrors the `patch` table from `db_technical_design.md`, extended with a
`patch_data` column storing the raw pixel bytes for each patch:

| Column       | Type         | Description                                                      |
|--------------|--------------|------------------------------------------------------------------|
| id           | BIGSERIAL PK | Sequential primary key (matches patch_id concept)                |
| patch_uid    | INT          | Unique patch identifier                                          |
| gt_label     | INT          | Ground truth label (0–9, random)                                 |
| event_ts     | TIMESTAMPTZ  | Timestamp when ground truth was set                              |
| image_id     | INT          | Foreign-key-like reference to source image (1–100)               |
| working_mag  | FLOAT        | Working magnification level (1.0–4.0, random)                    |
| patch_data   | BYTEA        | Raw patch pixel bytes — (32, 32, 3) uint8 array = 3,072 bytes    |

**Table type**: `UNLOGGED` — avoids WAL overhead during seeding; acceptable for a benchmark table.

**Index**: `PRIMARY KEY` B-tree index on `id` (matching production `patch` table index).

**Row count**: 1,000,000 (1M rows)

**Seeding method**: Client-side `INSERT` in 10k-row batches using `executemany` with
`numpy`-generated random uint8 patch data converted to bytes.

## Connection
Reads from environment variables `DB_HOST`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`;
falls back to prototyping defaults if not set.

In [1]:
import os
import time
import numpy as np
import psycopg2
from psycopg2 import extras

# ---------------------------------------------------------------------------
# Connection parameters — read from env vars, fall back to prototyping defaults
# ---------------------------------------------------------------------------
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TABLE       = 'bench_patch_batch_read'
TOTAL       = 1_000_000
BATCH       = 10_000
PATCH_SHAPE = (32, 32, 3)  # uint8 — 3,072 bytes per patch

rng = np.random.default_rng(0)

conn = psycopg2.connect(DSN)
conn.autocommit = False
cur = conn.cursor()

# Report PG version
cur.execute('SELECT version();')
pg_ver = cur.fetchone()[0].split(',')[0]
print(f'Connected to {pg_ver}')

# ---------------------------------------------------------------------------
# Drop + recreate table (idempotent)
# ---------------------------------------------------------------------------
cur.execute(f'DROP TABLE IF EXISTS {TABLE};')
conn.commit()
print(f'Dropped existing {TABLE} (if present).')

cur.execute(f'''
    CREATE UNLOGGED TABLE {TABLE} (
        id          BIGSERIAL PRIMARY KEY,
        patch_uid   INT,
        gt_label    INT,
        event_ts    TIMESTAMPTZ,
        image_id    INT,
        working_mag FLOAT,
        patch_data  BYTEA
    );
''')
conn.commit()
print(f'Table {TABLE} created (with patch_data BYTEA column).')

# ---------------------------------------------------------------------------
# Seed 1M rows in 10k-row client-side batches
# Each row carries 3,072 bytes of random uint8 patch data (32x32x3)
# ---------------------------------------------------------------------------
t0 = time.time()
insert_sql = f'''
    INSERT INTO {TABLE} (patch_uid, gt_label, event_ts, image_id, working_mag, patch_data)
    VALUES (%s, %s, NOW() - (%s * INTERVAL '1 second'), %s, %s, %s)
'''

for start in range(0, TOTAL, BATCH):
    end = min(start + BATCH, TOTAL)
    n   = end - start

    patch_uids   = list(range(start + 1, end + 1))
    gt_labels    = rng.integers(0, 10, size=n).tolist()
    offsets_s    = rng.integers(0, 365 * 24 * 3600, size=n).tolist()
    image_ids    = rng.integers(1, 101, size=n).tolist()
    working_mags = (rng.random(n) * 3 + 1).tolist()
    patch_datas  = [
        rng.integers(0, 256, size=PATCH_SHAPE, dtype=np.uint8).tobytes()
        for _ in range(n)
    ]

    rows = list(zip(patch_uids, gt_labels, offsets_s, image_ids, working_mags, patch_datas))
    extras.execute_batch(cur, insert_sql, rows, page_size=1000)
    conn.commit()
    print(f'  Seeded rows {start + 1} \u2013 {end}')

elapsed = time.time() - t0
print(f'Seeding complete in {elapsed:.1f}s')

# ---------------------------------------------------------------------------
# Verify
# ---------------------------------------------------------------------------
cur.execute(f'SELECT COUNT(*) FROM {TABLE};')
count = cur.fetchone()[0]
print(f'Final row count: {count:,}')
assert count == TOTAL, f'Expected {TOTAL} rows, got {count}'

conn.close()

Connected to PostgreSQL 15.17 (Debian 15.17-1.pgdg13+1) on x86_64-pc-linux-gnu
Dropped existing bench_patch_batch_read (if present).
Table bench_patch_batch_read created (with patch_data BYTEA column).


  Seeded rows 1 – 10000


  Seeded rows 10001 – 20000


  Seeded rows 20001 – 30000


  Seeded rows 30001 – 40000


...

  Seeded rows 950001 – 960000


  Seeded rows 960001 – 970000


  Seeded rows 970001 – 980000


  Seeded rows 980001 – 990000


  Seeded rows 990001 – 1000000
Seeding complete in 103.6s
Final row count: 1,000,000


In [2]:
# ---------------------------------------------------------------------------
# TEARDOWN  — run this cell to clean up after benchmarking
# ---------------------------------------------------------------------------
import os
import psycopg2

DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

conn = psycopg2.connect(DSN)
conn.autocommit = True
cur = conn.cursor()
cur.execute('DROP TABLE IF EXISTS bench_patch_batch_read;')
conn.close()
print('Teardown: bench_patch_batch_read dropped.')

Teardown: bench_patch_batch_read dropped.
